# 17 · Fonctions d'activation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch17_activations/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch17_activations/03_notebook.ipynb`, créée par `python tools/start_chapter.py 17`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 17.Q1 à 17.Q12, 17.R1 à 17.R3, 17.1, 17.2, 17.4, 17.5, 17.6, 17.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ 🧮 | ★ à ★★ | |
| A | 17.11 | Un MLP sans activation sur les lunes : quelle frontière ? | 🔮 | ★ | 15 |
|  | 17.12 | identity, step, relu, leaky_relu et leurs dérivées | 🔨 | ★ | 15 |
|  | 17.13 | Sigmoïde, tanh et softplus sans débordement | 🔨 | ★★ | 25 |
|  | 17.14 | ELU, SiLU, GELU et le registre get_activation | 🔨 | ★★ | 25 |
| B | 17.15 | Softmax et log_softmax stables (astuce log-sum-exp) | 🔨 | ★★ | 25 |
|  | 17.16 | La Jacobienne de la softmax en NumPy | 🔨 | ★★★ | 30 |
|  | 17.17 | Chasse au bug : nan, inf et mauvais axe | 🐛 | ★★ | 25 |
| C | 17.18 | Reproduire la galerie de la figure 17.22, dérivées comprises | 🎨 | ★★ | 25 |
|  | 17.19 | Softmax et température : prédire la distribution | 🔮 | ★★ | 15 |
|  | 17.20 | Quatre activations de scikit-learn sur les lunes et les spirales | 📦 | ★★ | 25 |
|  | 17.21 | Diagnostiquer la saturation sur des histogrammes de pré-activations | 📈 | ★★ | 20 |
| D | 17.22 | Une batterie de tests paramétrés pour toutes les activations | 🛠️ | ★★ | 25 |
|  | 17.23 | Produit des dérivées sur 10 couches : sigmoïde, tanh, ReLU, GELU | 🔬 | ★★★ | 45 |
|  | 17.24 | Défi spirales : au moins 97 % avec au plus 2 000 paramètres | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")  # your own library

## Objectifs et rappel express

- Voir qu'un réseau sans activation ne vaut qu'une couche, et ce qu'une ReLU y change.
- Écrire et vérifier contre SciPy et PyTorch les fonctions de `mylearn.nn.activations` : dix activations et leurs dérivées, `get_activation`, `softmax`, `log_softmax` et `softmax_derivative`, sans aucun débordement.
- Comparer les activations : saturation, température, choix pour un problème donné, gradients à travers dix couches.
- Tester toute une famille de fonctions avec des tests paramétrés et des tests de propriétés.

**Rappel express.** $\mathrm{ReLU}(z) = \max(0, z)$, $\mathrm{ReLU}'(0) = 0$ (PyTorch) ; $\sigma(z) = 1/(1 + e^{-z})$, $\sigma' = \sigma(1 - \sigma)$ ; $\tanh' = 1 - \tanh^2$ ; $\mathrm{softplus}(z) = \log(1 + e^z)$, $\mathrm{softplus}' = \sigma$ ; $\mathrm{ELU}(z) = \alpha(e^z - 1)$ pour $z \le 0$ ; $\mathrm{SiLU}(z) = z\,\sigma(z)$ ; $\mathrm{GELU}(z) = z\,\Phi(z)$, $\Phi(z) = \frac{1}{2}(1 + \mathrm{erf}(z/\sqrt{2}))$ ; $p_i = e^{z_i}/\sum_j e^{z_j}$, $\partial p_i/\partial z_j = p_i(\delta_{ij} - p_j)$ ; $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$. En NumPy : `np.exp(-np.abs(z))`, `np.log1p`, `np.expm1`, `np.where` (qui calcule les deux branches), `z.max(axis=-1, keepdims=True)`, `np.vectorize(math.erf)` ; en SciPy : `special.expit`, `special.softmax`, `special.log_softmax`, `special.erf`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 17.1, 17.2, 17.5 et 17.6, ∂ 17.4 et 🧮 17.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix ou une suite de codes : les caractères collés, dans l'ordre demandé (`"XYZ"`) ; une forme NumPy : entre guillemets (`"(5, 7)"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 17.3 et 17.7, les démonstrations de ∂ 17.4, la réflexion (🗣️ 17.9, 📄 17.10) et l'entretien se corrigent avec `05_solutions.md`.

**17.Q1 — À quoi sert la fonction d'activation ?**

In [ ]:
# 17.Q1: the values you found on paper
answer_17_Q1a = ...  # a) a whole number
answer_17_Q1b = ...  # b) True or False
answer_17_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_17_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_Q1a, answer_17_Q1b, answer_17_Q1c, answer_17_Q1d]):
    wb.check(f"17.Q1{letter}", answer)

**17.Q2 — Un nombre en entrée, un nombre en sortie : la forme d'une activation**

In [ ]:
# 17.Q2: the values you found on paper
answer_17_Q2a = ...  # a) a shape between quotes, e.g. "(5, 7)"
answer_17_Q2b = ...  # b) True or False
answer_17_Q2c = ...  # c) the letter of your choice, e.g. "E"
answer_17_Q2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_Q2a, answer_17_Q2b, answer_17_Q2c, answer_17_Q2d]):
    wb.check(f"17.Q2{letter}", answer)

**17.Q3 — Linéaire ou non linéaire ? Classer six courbes**

In [ ]:
# 17.Q3: the values you found on paper
answer_17_Q3a = ...  # a) six letters L or N, glued, e.g. "NNLLNL"

for letter, answer in zip("a", [answer_17_Q3a]):
    wb.check(f"17.Q3{letter}", answer)

**17.Q4 — Pourquoi une activation doit-elle donner une seule valeur par entrée ?**

In [ ]:
# 17.Q4: the values you found on paper
answer_17_Q4a = ...  # a) True or False
answer_17_Q4b = ...  # b) True or False
answer_17_Q4c = ...  # c) letters glued in alphabetical order, e.g. "BD"

for letter, answer in zip("abc", [answer_17_Q4a, answer_17_Q4b, answer_17_Q4c]):
    wb.check(f"17.Q4{letter}", answer)

**17.Q5 — Marche unité, Heaviside, signe : les distinguer**

In [ ]:
# 17.Q5: the values you found on paper
answer_17_Q5a = ...  # a) a number
answer_17_Q5b = ...  # b) a number
answer_17_Q5c = ...  # c) a number
answer_17_Q5d = ...  # d) a number
answer_17_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_17_Q5a, answer_17_Q5b, answer_17_Q5c, answer_17_Q5d, answer_17_Q5e]):
    wb.check(f"17.Q5{letter}", answer)

**17.Q6 — ReLU, leaky, paramétrique, décalée : qui fait quoi ?**

In [ ]:
# 17.Q6: the values you found on paper
answer_17_Q6a = ...  # a) a number
answer_17_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_17_Q6c = ...  # c) a number
answer_17_Q6d = ...  # d) True or False
answer_17_Q6e = ...  # e) a number

for letter, answer in zip("abcde", [answer_17_Q6a, answer_17_Q6b, answer_17_Q6c, answer_17_Q6d, answer_17_Q6e]):
    wb.check(f"17.Q6{letter}", answer)

**17.Q7 — Maxout et ReLU bruitée**

In [ ]:
# 17.Q7: the values you found on paper
answer_17_Q7a = ...  # a) a number
answer_17_Q7b = ...  # b) a number
answer_17_Q7c = ...  # c) a whole number
answer_17_Q7d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_Q7a, answer_17_Q7b, answer_17_Q7c, answer_17_Q7d]):
    wb.check(f"17.Q7{letter}", answer)

**17.Q8 — Saturation : quelles fonctions, et de quel côté ?**

In [ ]:
# 17.Q8: the values you found on paper
answer_17_Q8a = ...  # a) six characters among G, D, 2, 0, glued, e.g. "D20G2D"
answer_17_Q8b = ...  # b) a number with 4 decimals
answer_17_Q8c = ...  # c) a number with 4 decimals
answer_17_Q8d = ...  # d) a number with 2 decimals

for letter, answer in zip("abcd", [answer_17_Q8a, answer_17_Q8b, answer_17_Q8c, answer_17_Q8d]):
    wb.check(f"17.Q8{letter}", answer)

**17.Q9 — Swish, SiLU, GELU : des ReLU lissées**

In [ ]:
# 17.Q9: the values you found on paper
answer_17_Q9a = ...  # a) a number with 4 decimals
answer_17_Q9b = ...  # b) True or False
answer_17_Q9c = ...  # c) a number with 4 decimals
answer_17_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_17_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_17_Q9a, answer_17_Q9b, answer_17_Q9c, answer_17_Q9d, answer_17_Q9e]):
    wb.check(f"17.Q9{letter}", answer)

**17.Q10 — Reconnaître les courbes de la figure 17.22**

In [ ]:
# 17.Q10: the values you found on paper
answer_17_Q10a = ...  # a) six letters glued, e.g. "FEDCBA"

for letter, answer in zip("a", [answer_17_Q10a]):
    wb.check(f"17.Q10{letter}", answer)

**17.Q11 — Softmax : somme à 1, ordre conservé, mais pas une activation**

In [ ]:
# 17.Q11: the values you found on paper
answer_17_Q11a = ...  # a) True or False
answer_17_Q11b = ...  # b) True or False
answer_17_Q11c = ...  # c) True or False
answer_17_Q11d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_17_Q11a, answer_17_Q11b, answer_17_Q11c, answer_17_Q11d]):
    wb.check(f"17.Q11{letter}", answer)

**17.Q12 — Vrai ou faux : des scores 0,6 / 0,3 / 0,1 se lisent comme des probabilités**

In [ ]:
# 17.Q12: the values you found on paper
answer_17_Q12a = ...  # a) True or False
answer_17_Q12b = ...  # b) a number with 2 decimals
answer_17_Q12c = ...  # c) a number with 3 decimals
answer_17_Q12d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_Q12a, answer_17_Q12b, answer_17_Q12c, answer_17_Q12d]):
    wb.check(f"17.Q12{letter}", answer)

**17.R1 — Quelle initialisation va avec ReLU, et pourquoi ?**

In [ ]:
# 17.R1: the values you found on paper
answer_17_R1a = ...  # a) a number
answer_17_R1b = ...  # b) a number
answer_17_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_17_R1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_R1a, answer_17_R1b, answer_17_R1c, answer_17_R1d]):
    wb.check(f"17.R1{letter}", answer)

**17.R2 — D'où viennent les probabilités d'une forêt aléatoire ?**

In [ ]:
# 17.R2: the values you found on paper
answer_17_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_17_R2b = ...  # b) a number with 2 decimals
answer_17_R2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_17_R2a, answer_17_R2b, answer_17_R2c]):
    wb.check(f"17.R2{letter}", answer)

**17.R3 — Pourquoi un perceptron à seuil ne s'entraîne pas par gradient ?**

In [ ]:
# 17.R3: the values you found on paper
answer_17_R3a = ...  # a) a number
answer_17_R3b = ...  # b) True or False
answer_17_R3c = ...  # c) letters glued in alphabetical order, e.g. "BD"

for letter, answer in zip("abc", [answer_17_R3a, answer_17_R3b, answer_17_R3c]):
    wb.check(f"17.R3{letter}", answer)

**Ex 17.1 — Effondrer un réseau linéaire en un seul neurone**

In [ ]:
# Ex 17.1: the values you found on paper
answer_17_1a = ...  # a) a list of three numbers
answer_17_1b = ...  # b) a number
answer_17_1c = ...  # c) a number
answer_17_1d = ...  # d) a number
answer_17_1e = ...  # e) a whole number
answer_17_1f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_17_1a, answer_17_1b, answer_17_1c, answer_17_1d, answer_17_1e, answer_17_1f]):
    wb.check(f"17.1{letter}", answer)

**Ex 17.2 — Tableau de valeurs de neuf activations**

In [ ]:
# Ex 17.2: the values you found on paper
answer_17_2a = ...  # a) a list of nine numbers, 4 decimals
answer_17_2b = ...  # b) a list of nine numbers, 4 decimals
answer_17_2c = ...  # c) a whole number

for letter, answer in zip("abc", [answer_17_2a, answer_17_2b, answer_17_2c]):
    wb.check(f"17.2{letter}", answer)

**Ex 17.4 — Dériver la sigmoïde, tanh et softplus**

In [ ]:
# Ex 17.4: the values you found on paper
answer_17_4a = ...  # a) a number
answer_17_4b = ...  # b) a number
answer_17_4c = ...  # c) a number
answer_17_4d = ...  # d) a number with 5 decimals
answer_17_4e = ...  # e) a number
answer_17_4f = ...  # f) a number with 4 decimals

for letter, answer in zip("abcdef", [answer_17_4a, answer_17_4b, answer_17_4c, answer_17_4d, answer_17_4e, answer_17_4f]):
    wb.check(f"17.4{letter}", answer)

**Ex 17.5 — Maxout et ReLU : construire |x|, une cuvette et une bosse**

In [ ]:
# Ex 17.5: the values you found on paper
answer_17_5a = ...  # a) a list of two numbers, in increasing order
answer_17_5b = ...  # b) a number
answer_17_5c = ...  # c) a list [a, b, c]
answer_17_5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_17_5a, answer_17_5b, answer_17_5c, answer_17_5d]):
    wb.check(f"17.5{letter}", answer)

**Ex 17.6 — Softmax à la main : translation, température, rapports**

In [ ]:
# Ex 17.6: the values you found on paper
answer_17_6a = ...  # a) a number with 2 decimals
answer_17_6b = ...  # b) a number with 3 decimals
answer_17_6c = ...  # c) a number with 3 decimals
answer_17_6d = ...  # d) a number with 3 decimals
answer_17_6e = ...  # e) a number with 3 decimals
answer_17_6f = ...  # f) a number with 2 decimals

for letter, answer in zip("abcdef", [answer_17_6a, answer_17_6b, answer_17_6c, answer_17_6d, answer_17_6e, answer_17_6f]):
    wb.check(f"17.6{letter}", answer)

**Ex 17.8 — Fermi : une activation coûte-t-elle cher face au produit matriciel ?**

In [ ]:
# Ex 17.8: the values you found on paper
answer_17_8a = ...  # a) a whole number of millions
answer_17_8b = ...  # b) a whole number of thousands
answer_17_8c = ...  # c) a whole number
answer_17_8d = ...  # d) a percentage with 1 decimal
answer_17_8e = ...  # e) a number of MB, 1 decimal
answer_17_8f = ...  # f) a number of microseconds, 1 decimal
answer_17_8g = ...  # g) a number of microseconds, 1 decimal

for letter, answer in zip("abcdefg", [answer_17_8a, answer_17_8b, answer_17_8c, answer_17_8d, answer_17_8e, answer_17_8f, answer_17_8g]):
    wb.check(f"17.8{letter}", answer)

## Partie A · Les activations élément par élément

Prédire ce que devient un MLP sans activation (17.11), puis écrire, avec leurs dérivées, les dix activations de ta librairie : les fonctions linéaires par morceaux (17.12), les fonctions lisses de base, sans débordement (17.13), puis ELU, SiLU, GELU et le registre `get_activation` (17.14). La cellule suivante définit les outils des parties A à D. Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie. Après un redémarrage du noyau, exécute la cellule de configuration, puis la cellule d'outils qui suit (`# Tools for the notebook exercises`), avant toute vérification.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import contextlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import special
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from wb.errors import MylearnMissingError

try:
    import torch
except ImportError:                       # the comparisons with PyTorch are then left to the tests
    torch = None


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple)) else value
    except (TypeError, ValueError):
        shown = value
    print(f"{ex_id}:", shown)


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that the tests run after it can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


def quiet(f, *args, **kwargs):
    """Call f and return (result, the RuntimeWarnings it emitted: overflow, invalid value, division by zero)."""
    with warnings.catch_warnings(record=True) as caught, np.errstate(over="warn", invalid="warn", divide="warn",
                                                                     under="ignore"):
        warnings.simplefilter("always")
        result = f(*args, **kwargs)
    return result, [str(w.message) for w in caught if issubclass(w.category, RuntimeWarning)]


def same(ex_id, label, got, want, rtol=1e-9, atol=1e-12):
    """✅ if `got` is an array equal to `want` up to rounding, ❌ with both values otherwise; returns the verdict."""
    if got is None:
        print(f"❌ Ex {ex_id} : {label} renvoie None : as-tu oublié le return ?")
        return False
    try:
        arr = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        print(f"❌ Ex {ex_id} : {label} : attendu un tableau de nombres, reçu un objet {type(got).__name__}.")
        return False
    want = np.asarray(want, dtype=float)
    if arr.shape != want.shape:
        print(f"❌ Ex {ex_id} : {label} : attendu la forme {want.shape}, reçu la forme {arr.shape}.")
        return False
    ok = bool(np.allclose(arr, want, rtol=rtol, atol=atol, equal_nan=True))
    shown = np.round(arr, 8).tolist() if arr.size <= 6 else f"un tableau {arr.shape}"
    if ok or want.size <= 6:
        failure = f"{label} : attendu {np.round(want, 8).tolist()}, reçu {shown}."
    else:
        bad = np.argwhere(~np.isclose(arr, want, rtol=rtol, atol=atol, equal_nan=True))[0]
        at = tuple(int(i) for i in bad) if arr.ndim > 1 else int(bad[0])
        failure = (f"{label} : première différence à l'indice {at} : attendu {float(want[tuple(bad)]):.8g}, reçu "
                   f"{float(arr[tuple(bad)]):.8g} (plus grand écart {np.nanmax(np.abs(arr - want)):.3g}).")
    verdict(ex_id, ok, f"{label} : {shown}, comme attendu.", failure)
    return ok


def fit_quietly(model, X, y):
    """Fit a scikit-learn model without its ConvergenceWarning (the number of iterations is part of the protocol)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return model.fit(X, y)


GRID = np.arange(-600, 601) / 100.0                      # z from -6 to 6, with an exact 0.0
EXTREME = np.array([-1000.0, -700.0, -40.0, 0.0, 40.0, 700.0, 1000.0])
TEST_FILES = {"activations": "tests/test_ch17_activations.py"}


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn.nn.<module> selected by `keyword` (on YOUR code by default)."""
    test_file = TEST_FILES[module]
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    try:
        result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=300,
                                env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    except subprocess.TimeoutExpired:
        print("❌ les tests n'ont pas fini en 5 min : une boucle de ton code ne s'arrête-t-elle pas ?")
        return
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une fonction oublie-t-elle son return ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


print("tools ready · PyTorch", "available" if torch is not None
      else "absent (the PyTorch oracle tests are skipped: run them on Colab; the other tests still run)")

### Ex 17.11 — Un MLP sans activation sur les lunes : quelle frontière ? 🔮 ★ ⏱️ 15 min
**Objectif :** prédire ce que vaut un réseau profond sans activation, face à la régression logistique et au même réseau avec ReLU.  
**Prérequis :** ✏️ 17.1 · ch. 15 (`MLPClassifier`, les demi-lunes) · fiche §17.2 · **Fil rouge :** synthétique · **Parcours :** R, C

Sur deux demi-lunes, comme au ch. 15 (500 points, bruit 0,2 ; 40 % pour le test), on compare trois modèles : une régression logistique (3 paramètres), un `MLPClassifier` à deux couches cachées de 100 neurones **sans activation** (`activation="identity"`, plus de 10 000 paramètres) et le même réseau avec ReLU. Les deux MLP sont entraînés par `solver="lbfgs"` (une méthode de second ordre, bien adaptée aux petits datasets), avec `random_state=0`.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `beats_lr_11` : le MLP sans activation a-t-il une accuracy de test **supérieure de plus de 2 points** (0,02) à celle de la régression logistique ? (`True` ou `False`) ;
b) `agree_11` : sur quelle part des points de test le MLP sans activation et la régression logistique prédisent-ils la même classe ? (A) moins de 80 % ; (B) de 80 % à 95 % ; (C) plus de 95 % ;
c) `relu_11` : l'accuracy de test du MLP avec ReLU : (A) moins de 0,85 ; (B) de 0,85 à 0,90 ; (C) plus de 0,90.

Puis exécute l'expérience : elle entraîne les trois modèles, affiche leurs accuracies, leur accord, la direction des poids « effondrés » du MLP sans activation et les trois frontières de décision, et vérifie tes prédictions.

In [ ]:
X_11, y_11 = wb.synth.make_moons(n=500, noise=0.2, seed=1711)
Xtr_11, Xte_11, ytr_11, yte_11 = train_test_split(X_11, y_11, test_size=0.4, random_state=1711, stratify=y_11)
print("train", Xtr_11.shape, "test", Xte_11.shape)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
beats_lr_11 = ...   # a) True or False
agree_11 = ...      # b) "A", "B" or "C"
relu_11 = ...       # c) "A", "B" or "C"

In [ ]:
if any(answer is ... or answer is None for answer in [beats_lr_11, agree_11, relu_11]):
    print("⏳ Ex 17.11 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    logreg_11 = fit_quietly(LogisticRegression(), Xtr_11, ytr_11)
    flat_11 = fit_quietly(MLPClassifier((100, 100), activation="identity", solver="lbfgs", max_iter=3000, random_state=0),
                          Xtr_11, ytr_11)
    bent_11 = fit_quietly(MLPClassifier((100, 100), activation="relu", solver="lbfgs", max_iter=3000, random_state=0),
                          Xtr_11, ytr_11)
    n_params_11 = sum(W.size for W in flat_11.coefs_) + sum(b.size for b in flat_11.intercepts_)
    acc_11 = {name: model.score(Xte_11, yte_11) for name, model in
              [("logistic regression", logreg_11), ("MLP without activation", flat_11), ("MLP with ReLU", bent_11)]}
    for name, acc in acc_11.items():
        print(f"{name:24s} test accuracy {acc:.3f}")
    agreement_11 = float(np.mean(flat_11.predict(Xte_11) == logreg_11.predict(Xte_11)))
    print(f"MLP without activation: {n_params_11} parameters · same class as the logistic regression on {agreement_11:.1%} of the test points")
    W_eq_11 = flat_11.coefs_[0] @ flat_11.coefs_[1] @ flat_11.coefs_[2]       # the collapsed weights (2, 1)
    print("direction of the collapsed weights:", np.round(W_eq_11.ravel() / np.linalg.norm(W_eq_11), 3).tolist(),
          "· of the logistic regression:", np.round(logreg_11.coef_.ravel() / np.linalg.norm(logreg_11.coef_), 3).tolist())
    fig_11, axes_11 = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, (name, model) in zip(axes_11, [("régression logistique", logreg_11), ("MLP sans activation", flat_11),
                                           ("MLP avec ReLU", bent_11)]):
        wb.plot.plot_decision_boundary(model, Xte_11, yte_11, ax=ax, title=name)
    plt.tight_layout()
    plt.show()
    wb.check("17.11a", beats_lr_11)
    wb.check("17.11b", agree_11)
    wb.check("17.11c", relu_11)

Dans tes notes : combien de nombres suffisent à décrire la frontière du MLP sans activation, malgré ses milliers de paramètres ? Compare les deux directions de poids affichées : d'où peuvent venir les écarts entre ce MLP et la régression logistique ? Qu'est-ce que la ReLU a changé à la frontière ?

### Ex 17.12 — identity, step, relu, leaky_relu et leurs dérivées 🔨 ★ ⏱️ 15 min
**Objectif :** écrire les activations linéaires par morceaux et leurs dérivées, avec la convention de PyTorch aux coudes.  
**Prérequis :** ✏️ 17.2 · fiche §17.3.1, §17.4, §17.5 (encadré sur la dérivée en un point anguleux) · **Fil rouge :** synthétique · **mylearn :** `nn/activations.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/activations.py` (créé par `python tools/start_chapter.py 17`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement (`math.erf` pour GELU) ; SciPy et PyTorch sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à SciPy, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch17_activations.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible). Les tests sont écrits comme te le fera faire 🛠️ 17.22 : un test paramétré par propriété, exécuté sur toute la famille (`[relu]`, `[softplus-beta2]`, `[gelu-approx_derivative]`…).

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : chaque fonction rend un **nouveau** tableau `float64` de la forme de `z` (un tableau de dimension 0, ou un `np.float64`, pour un nombre seul), jamais une vue de `z`, qu'elle laisse intact ; une liste d'entiers est acceptée ; aucune fonction n'émet de `RuntimeWarning` (débordement, valeur invalide, division par zéro) ni ne rend de valeur infinie pour des entrées entre $-1\,000$ et $1\,000$ ; chaque dérivée prend la **pré-activation** `z` et vaut, aux points anguleux, la valeur de PyTorch.

Écris `identity`, `step`, `relu` et `leaky_relu`, et leurs quatre dérivées `..._derivative`. **Contrat** propre à ces fonctions : `step(z, threshold, low, high)` rend `high` dès que $z \ge$ `threshold` et `low` strictement en dessous ; sa dérivée est nulle partout ; $\mathrm{ReLU}'(0) = 0$ et $\mathrm{leaky}'(0) =$ `negative_slope`, comme PyTorch ; la pente par défaut de `leaky_relu` est 0,01.

La vérification essaie les exemples de la docstring, compare `relu` et `step` à NumPy sur une grille de $-6$ à $6$, vérifie que `identity` rend une copie, puis lance les tests de l'exercice (`-k "identity or step or relu"`).

Dans tes notes : pourquoi `identity` ne peut-elle pas se contenter de `return np.asarray(z, dtype=float)` ? Quel test de la batterie le montre ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")   # reload your saved file
with checked("17.12"):
    A = mylearn.nn.activations
    same("17.12", "identity([-2, 0, 2])", A.identity([-2, 0, 2]), [-2.0, 0.0, 2.0])
    same("17.12", "step([-2, 0, 2])", A.step([-2.0, 0.0, 2.0]), [0.0, 1.0, 1.0])
    same("17.12", "step([-2, 0, 2], low=-1)", A.step([-2.0, 0.0, 2.0], low=-1.0), [-1.0, 1.0, 1.0])
    same("17.12", "relu_derivative([-2, 0, 2])", A.relu_derivative([-2.0, 0.0, 2.0]), [0.0, 0.0, 1.0])
    same("17.12", "leaky_relu([-2, 0, 2], negative_slope=0.1)", A.leaky_relu([-2.0, 0.0, 2.0], negative_slope=0.1), [-0.2, 0.0, 2.0])
    same("17.12", "leaky_relu_derivative([-2, 0, 2], negative_slope=0.1)",
         A.leaky_relu_derivative([-2.0, 0.0, 2.0], negative_slope=0.1), [0.1, 0.1, 1.0])
with checked("17.12"):
    A = mylearn.nn.activations
    same("17.12", "relu sur z de -6 à 6, comme np.maximum", A.relu(GRID), np.maximum(GRID, 0.0))
    same("17.12", "step(z, threshold=0.5, low=-1, high=2) sur la grille", A.step(GRID, threshold=0.5, low=-1.0, high=2.0),
         -1.0 + 3.0 * np.heaviside(GRID - 0.5, 1.0))
    z_12 = np.array([1.0, 2.0, 3.0])
    out_12 = A.identity(z_12)
    verdict("17.12", isinstance(out_12, np.ndarray) and not np.shares_memory(out_12, z_12),
            "identity rend un nouveau tableau (le modifier ne touche pas z).",
            "identity(z) partage la mémoire de z : modifier le résultat modifierait z (np.asarray ne copie pas un tableau float64).")
if written(lambda: mylearn.nn.activations.relu([1.0])):
    run_mylearn_tests("activations", keyword="identity or step or relu")
else:
    print("⏳ Ex 17.12 : les tests attendent tes fonctions.")

### Ex 17.13 — Sigmoïde, tanh et softplus sans débordement 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les fonctions lisses de base et leurs dérivées sans aucun débordement, même pour des entrées de ±1 000.  
**Prérequis :** Ex 17.12 · fiche §17.6, section sur le calcul sans débordement · ch. 13 (sigmoïde stable) · **Fil rouge :** synthétique · **mylearn :** `nn/activations.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/activations.py` (créé par `python tools/start_chapter.py 17`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement (`math.erf` pour GELU) ; SciPy et PyTorch sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à SciPy, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch17_activations.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible). Les tests sont écrits comme te le fera faire 🛠️ 17.22 : un test paramétré par propriété, exécuté sur toute la famille (`[relu]`, `[softplus-beta2]`, `[gelu-approx_derivative]`…).

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : chaque fonction rend un **nouveau** tableau `float64` de la forme de `z` (un tableau de dimension 0, ou un `np.float64`, pour un nombre seul), jamais une vue de `z`, qu'elle laisse intact ; une liste d'entiers est acceptée ; aucune fonction n'émet de `RuntimeWarning` (débordement, valeur invalide, division par zéro) ni ne rend de valeur infinie pour des entrées entre $-1\,000$ et $1\,000$ ; chaque dérivée prend la **pré-activation** `z` et vaut, aux points anguleux, la valeur de PyTorch.

Écris `sigmoid`, `tanh` et `softplus`, et leurs trois dérivées. **Contrat** propre à ces fonctions : `sigmoid(-1000)` vaut 0 et `sigmoid(1000)` vaut 1, sans avertissement, et les petites probabilités sont gardées (`sigmoid(-700)` est comparé à SciPy avec une tolérance **relative**) ; `softplus(z, beta, threshold)` rend `z` lui-même seulement quand `beta * z` est **strictement** supérieur à `threshold` (convention de PyTorch), et $\log(1 + e^{\beta z})/\beta$ sinon, seuil compris ; sa dérivée vaut exactement 1 au-dessus du seuil.

La vérification essaie les exemples de la docstring, appelle chaque fonction sur des valeurs extrêmes en guettant les avertissements, compare `sigmoid(-700)` et `softplus` à SciPy et NumPy, puis lance les tests (`-k "sigmoid or tanh or softplus"`).

Dans tes notes : quelle est la plus petite valeur strictement positive que rend ta sigmoïde, et pour quel $z$ environ ? Pourquoi la forme $1 - 1/(1 + e^{z})$, mathématiquement égale à $\sigma(z)$, ne la garderait-elle pas ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")   # reload your saved file
with checked("17.13"):
    A = mylearn.nn.activations
    same("17.13", "sigmoid([-2, 0, 2])", A.sigmoid([-2.0, 0.0, 2.0]), special.expit([-2.0, 0.0, 2.0]))
    same("17.13", "sigmoid_derivative([-2, 0, 2])", A.sigmoid_derivative([-2.0, 0.0, 2.0]), [0.10499358540350652, 0.25, 0.10499358540350652])
    same("17.13", "tanh_derivative([-2, 0, 2])", A.tanh_derivative([-2.0, 0.0, 2.0]), 1 - np.tanh([-2.0, 0.0, 2.0]) ** 2)
    same("17.13", "softplus([-2, 0, 2])", A.softplus([-2.0, 0.0, 2.0]), np.logaddexp(0.0, [-2.0, 0.0, 2.0]))
    same("17.13", "softplus_derivative([-2, 0, 2])", A.softplus_derivative([-2.0, 0.0, 2.0]), special.expit([-2.0, 0.0, 2.0]))
with checked("17.13"):
    A = mylearn.nn.activations
    for name_13 in ["sigmoid", "sigmoid_derivative", "tanh", "tanh_derivative", "softplus", "softplus_derivative"]:
        values_13, alarms_13 = quiet(getattr(A, name_13), EXTREME.copy())
        finite_13 = bool(np.all(np.isfinite(np.asarray(values_13, dtype=float))))
        verdict("17.13", finite_13 and not alarms_13, f"{name_13} sur z = ±1000, ±700, ±40, 0 : fini et sans avertissement.",
                f"{name_13} sur z = ±1000, ±700, ±40, 0 : " + (f"avertissement « {alarms_13[0]} »" if alarms_13 else "une valeur infinie ou nan")
                + " (relis le premier principe de la section sur le calcul sans débordement).")
    same("17.13", "sigmoid(-700) (tolérance relative)", A.sigmoid([-700.0]), special.expit([-700.0]), rtol=1e-6, atol=0.0)
    same("17.13", "softplus(z, beta=2, threshold=10) sur la grille (seuil compris, en z = 5)", A.softplus(GRID, beta=2.0, threshold=10.0),
         np.where(2 * GRID > 10, GRID, np.logaddexp(0.0, 2 * GRID) / 2), rtol=1e-9, atol=0.0)
if written(lambda: mylearn.nn.activations.sigmoid([1.0])):
    run_mylearn_tests("activations", keyword="sigmoid or tanh or softplus")
else:
    print("⏳ Ex 17.13 : les tests attendent tes fonctions.")

### Ex 17.14 — ELU, SiLU, GELU et le registre get_activation 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les ReLU lissées, GELU exacte et approchée, et le registre qui associe un nom à un couple (f, f′).  
**Prérequis :** Ex 17.13 · fiche §17.6 (encadré sur Φ et erf), §17.7 · **Fil rouge :** synthétique · **mylearn :** `nn/activations.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/activations.py` (créé par `python tools/start_chapter.py 17`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement (`math.erf` pour GELU) ; SciPy et PyTorch sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à SciPy, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch17_activations.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible). Les tests sont écrits comme te le fera faire 🛠️ 17.22 : un test paramétré par propriété, exécuté sur toute la famille (`[relu]`, `[softplus-beta2]`, `[gelu-approx_derivative]`…).

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : chaque fonction rend un **nouveau** tableau `float64` de la forme de `z` (un tableau de dimension 0, ou un `np.float64`, pour un nombre seul), jamais une vue de `z`, qu'elle laisse intact ; une liste d'entiers est acceptée ; aucune fonction n'émet de `RuntimeWarning` (débordement, valeur invalide, division par zéro) ni ne rend de valeur infinie pour des entrées entre $-1\,000$ et $1\,000$ ; chaque dérivée prend la **pré-activation** `z` et vaut, aux points anguleux, la valeur de PyTorch.

Écris `elu`, `silu` et `gelu`, leurs dérivées, et `get_activation`. **Contrat** propre à ces fonctions : $\mathrm{ELU}'(0) = \alpha$ (PyTorch) ; `gelu` et `gelu_derivative` acceptent `approximate="none"` (la formule exacte, avec `math.erf` vectorisé) ou `approximate="tanh"` (l'approximation de la fiche), et lèvent `ValueError` pour toute autre valeur, `None` compris ; `get_activation(name)` ignore la casse, connaît l'alias `"swish"` de `"silu"`, rend un couple fait des fonctions **du module elles-mêmes** (`get_activation("relu")` est `(relu, relu_derivative)`), et lève `ValueError` pour un nom inconnu, avec un message qui donne les dix noms valides.

La vérification essaie les exemples de la docstring, compare `gelu` à $z\,\Phi(z)$ calculé avec `scipy.special.erf`, essaie des noms et une approximation inconnus, puis lance les tests (`-k "(elu or silu or gelu or get_activation) and not relu"`).

Dans tes notes : pourquoi `get_activation("softmax")` doit-il lever une erreur, alors que ta librairie a une fonction `softmax` ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")   # reload your saved file
with checked("17.14"):
    A = mylearn.nn.activations
    same("17.14", "elu([-2, 0, 2])", A.elu([-2.0, 0.0, 2.0]), [-0.8646647167633873, 0.0, 2.0])
    same("17.14", "elu_derivative([0], alpha=1.5)", A.elu_derivative([0.0], alpha=1.5), [1.5])
    same("17.14", "silu_derivative([-2, 0, 2])", A.silu_derivative([-2.0, 0.0, 2.0]), [-0.09078424878489121, 0.5, 1.0907842487848914])
    exact_14 = GRID * 0.5 * (1 + special.erf(GRID / np.sqrt(2)))
    same("17.14", "gelu sur la grille, comme z Φ(z) avec scipy.special.erf", A.gelu(GRID), exact_14)
    same("17.14", 'gelu([-2, 0, 2], approximate="tanh")', A.gelu([-2.0, 0.0, 2.0], approximate="tanh"),
         [-0.04540230591222698, 0.0, 1.954597694087773])
    same("17.14", 'gelu_derivative([-2, 0, 2], approximate="tanh")', A.gelu_derivative([-2.0, 0.0, 2.0], approximate="tanh"),
         [-0.08609925660415786, 0.5, 1.0860992566041579])
with checked("17.14"):
    A = mylearn.nn.activations
    try:
        A.gelu([1.0], approximate="exact")
        print('❌ Ex 17.14 : gelu(z, approximate="exact") devrait lever ValueError (les deux noms valides sont "none" et "tanh").')
    except ValueError as exc_14:
        print(f'✅ Ex 17.14 : gelu(z, approximate="exact") lève ValueError : {exc_14}')
    pair_14 = A.get_activation("ReLU")
    verdict("17.14", tuple(pair_14) == (A.relu, A.relu_derivative), 'get_activation("ReLU") rend (relu, relu_derivative), les fonctions du module.',
            f'get_activation("ReLU") rend {pair_14!r} : attendu (relu, relu_derivative), les fonctions du module elles-mêmes.')
    verdict("17.14", A.get_activation("swish")[0] is A.silu, '"swish" est un alias de silu.', 'get_activation("swish") doit rendre silu.')
    try:
        A.get_activation("sigmoïde")
        print('❌ Ex 17.14 : get_activation("sigmoïde") devrait lever ValueError.')
    except ValueError as exc_14:
        print(f"✅ Ex 17.14 : un nom inconnu lève ValueError : {exc_14}")
if written(lambda: mylearn.nn.activations.gelu([1.0])):
    run_mylearn_tests("activations", keyword="(elu or silu or gelu or get_activation) and not relu")
else:
    print("⏳ Ex 17.14 : les tests attendent tes fonctions.")

## Partie B · La softmax

La softmax stable et son logarithme (17.15), sa Jacobienne pour un batch (17.16), puis une chasse aux `nan` dans la sortie d'un petit réseau (17.17).

### Ex 17.15 — Softmax et log_softmax stables (astuce log-sum-exp) 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire une softmax et une log-softmax qui ne débordent jamais, le long de n'importe quel axe.  
**Prérequis :** Ex 17.13 · fiche §17.8, section sur le calcul sans débordement · ch. 13 (`logsumexp`) · **Fil rouge :** synthétique · **mylearn :** `nn/activations.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/activations.py` (créé par `python tools/start_chapter.py 17`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement (`math.erf` pour GELU) ; SciPy et PyTorch sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à SciPy, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch17_activations.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible). Les tests sont écrits comme te le fera faire 🛠️ 17.22 : un test paramétré par propriété, exécuté sur toute la famille (`[relu]`, `[softplus-beta2]`, `[gelu-approx_derivative]`…).

Écris `softmax` et `log_softmax`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) : les deux fonctions rendent un nouveau tableau `float64` de la forme de `z`, laissent `z` intact, et lèvent `ValueError` pour un nombre seul (0 dimension) ; `axis` désigne l'axe des classes (par défaut le dernier), et n'importe quel axe d'un tableau 1-D, 2-D ou 3-D doit marcher ; les probabilités sont positives et somment à 1 le long de `axis` ; aucun avertissement ni `nan` pour des logits de l'ordre de $\pm 1\,000$ (`softmax`) ou de $10^4$ (`log_softmax`, qui ne calcule jamais `log(0)`).

La vérification essaie les exemples de la docstring et des logits énormes, compare tes fonctions à `scipy.special.softmax` et `log_softmax` sur un batch le long des deux axes, essaie un nombre seul, puis lance les tests (`-k "softmax and not derivative"`).

Dans tes notes : la softmax de `[-1000, -1001, -1003]` sans soustraction du maximum, que donne-t-elle, et pourquoi ? Le soustraire change-t-il le résultat mathématique ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")   # reload your saved file
with checked("17.15"):
    A = mylearn.nn.activations
    same("17.15", "softmax([1, 2, 3])", A.softmax([1.0, 2.0, 3.0]), special.softmax([1.0, 2.0, 3.0]))
    big_15, alarms_15 = quiet(A.softmax, [1001.0, 1002.0, 1003.0])
    same("17.15", "softmax([1001, 1002, 1003])", big_15, special.softmax([1.0, 2.0, 3.0]))
    verdict("17.15", not alarms_15, "softmax de logits de l'ordre de 1 000 : aucun avertissement.",
            f"softmax([1001, 1002, 1003]) émet « {alarms_15[0] if alarms_15 else ''} » : soustrais d'abord le maximum.")
    logp_15, alarms_15 = quiet(A.log_softmax, [0.0, 10000.0])
    same("17.15", "log_softmax([0, 10000])", logp_15, [-10000.0, 0.0])
    verdict("17.15", not alarms_15, "log_softmax([0, 10000]) : aucun avertissement.",
            f"log_softmax([0, 10000]) émet « {alarms_15[0] if alarms_15 else ''} » : z − logsumexp(z), jamais log(softmax(z)).")
with checked("17.15"):
    A = mylearn.nn.activations
    batch_15 = np.random.default_rng(1715).normal(scale=3.0, size=(4, 6))
    same("17.15", "softmax d'un batch (4, 6), axe des classes -1", A.softmax(batch_15), special.softmax(batch_15, axis=-1))
    same("17.15", "softmax du même batch avec axis=0", A.softmax(batch_15, axis=0), special.softmax(batch_15, axis=0))
    same("17.15", "log_softmax du batch", A.log_softmax(batch_15), special.log_softmax(batch_15, axis=-1))
    try:
        A.softmax(3.0)
        print("❌ Ex 17.15 : softmax(3.0) devrait lever ValueError (un nombre seul n'a pas d'axe de classes).")
    except ValueError as exc_15:
        print(f"✅ Ex 17.15 : softmax d'un nombre seul lève ValueError : {exc_15}")
if written(lambda: mylearn.nn.activations.softmax([1.0, 2.0])):
    run_mylearn_tests("activations", keyword="softmax and not derivative")
else:
    print("⏳ Ex 17.15 : les tests attendent tes fonctions.")

### Ex 17.16 — La Jacobienne de la softmax en NumPy 🔨 ★★★ ⏱️ 30 min
**Objectif :** écrire la Jacobienne de la softmax pour un exemple et pour un batch, sans boucle sur les exemples.  
**Prérequis :** Ex 17.15 · ∂ 17.7 · fiche §17.8 (encadré sur la Jacobienne) · **Fil rouge :** synthétique · **mylearn :** `nn/activations.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/activations.py` (créé par `python tools/start_chapter.py 17`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement (`math.erf` pour GELU) ; SciPy et PyTorch sont les **oracles** des tests. La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à SciPy, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch17_activations.py -q` les lance tous dans un terminal (la docstring du module dit `pytest tests/ -q` : préfère cette commande du chapitre, plus courte et plus lisible). Les tests sont écrits comme te le fera faire 🛠️ 17.22 : un test paramétré par propriété, exécuté sur toute la famille (`[relu]`, `[softplus-beta2]`, `[gelu-approx_derivative]`…).

Écris `softmax_derivative`. **Contrat** : pour des logits de forme `(K,)`, la fonction rend la matrice `(K, K)` de case `[i, j]` $= \partial p_i / \partial z_j = p_i(\delta_{ij} - p_j)$ ; pour un batch `(n, K)`, un tableau `(n, K, K)`, une matrice par exemple ; `ValueError` pour toute autre dimension ; aucun avertissement pour des logits de l'ordre de $\pm 1\,000$ ; `z` reste intact.

La vérification essaie les exemples de la docstring, compare ta Jacobienne à des différences centrées de `scipy.special.softmax` sur un batch de 5 exemples, vérifie la symétrie et les lignes de somme nulle, puis lance les tests (`-k softmax_derivative`).

Dans tes notes : combien de nombres contient le résultat pour un batch de 64 exemples et un vocabulaire de 50 000 tokens ? Pourquoi aucune bibliothèque ne forme ce tableau pendant l'entraînement ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="17")   # reload your saved file
def numeric_jacobian_16(z, h=1e-6):
    """Central differences of scipy.special.softmax along the last axis: entry [..., i, j] = dp_i / dz_j."""
    K = z.shape[-1]
    J = np.empty(z.shape[:-1] + (K, K))
    for j in range(K):
        step = np.zeros(K)
        step[j] = h
        J[..., :, j] = (special.softmax(z + step, axis=-1) - special.softmax(z - step, axis=-1)) / (2 * h)
    return J


with checked("17.16"):
    A = mylearn.nn.activations
    same("17.16", "softmax_derivative([0, 0])", A.softmax_derivative([0.0, 0.0]), [[0.25, -0.25], [-0.25, 0.25]])
    z_16 = np.random.default_rng(1716).normal(scale=2.0, size=(5, 4))
    J_16 = np.asarray(A.softmax_derivative(z_16), dtype=float)
    verdict("17.16", J_16.shape == (5, 4, 4), "5 exemples de 4 logits : un tableau (5, 4, 4), une Jacobienne par exemple.",
            f"attendu la forme (5, 4, 4), reçu {J_16.shape}.")
    if J_16.shape == (5, 4, 4):
        same("17.16", "Jacobiennes du batch, comme les différences centrées de scipy.special.softmax", J_16,
             numeric_jacobian_16(z_16), rtol=1e-6, atol=1e-8)
        verdict("17.16", np.allclose(J_16, np.swapaxes(J_16, 1, 2)) and np.allclose(J_16.sum(axis=2), 0.0, atol=1e-12),
                "chaque Jacobienne est symétrique, et ses lignes somment à 0.",
                "une Jacobienne de la softmax est symétrique, et ses lignes somment à 0 : vérifie les axes de ton produit.")
if written(lambda: mylearn.nn.activations.softmax_derivative([0.0, 1.0])):
    run_mylearn_tests("activations", keyword="softmax_derivative")
else:
    print("⏳ Ex 17.16 : les tests attendent ta fonction.")

### Ex 17.17 — Chasse au bug : nan, inf et mauvais axe 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver quatre erreurs dans la sortie d'un petit réseau, dont trois qui ne se voient que sur des valeurs extrêmes.  
**Prérequis :** Ex 17.13 · Ex 17.15 · fiche §17.6 (encadré sur les dérivées), section sur le calcul sans débordement · **Parcours :** C

Un collègue a écrit `forward_17`, la passe avant d'un petit réseau tanh à une couche cachée : elle rend les probabilités des classes, la cross-entropy moyenne du batch, et la pente $\tanh'(z_1)$ de chaque neurone caché, que la rétropropagation du ch. 18 utilisera. Sur ses données de test, tout semble marcher. Il y a pourtant **quatre** erreurs : trouve-les et corrige-les dans la cellule ci-dessous, sans changer la signature ni ce que rend la fonction. Tu peux utiliser les fonctions de ta librairie (`mylearn.nn.activations`).

La vérification contrôle quatre propriétés, dans l'ordre où les erreurs se masquent (un point ne se teste que si les précédents passent) : 1) aucun débordement de l'exponentielle pour des logits de l'ordre de 1 000 ; 2) chaque ligne de probabilités somme à 1 et vaut la softmax de SciPy ; 3) une loss finie et juste quand la bonne classe d'un exemple a une probabilité qui s'arrondit à 0 ; 4) la pente vaut $\tanh'(z_1)$. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque erreur, quel symptôme aurait-elle eu pendant un vrai entraînement ? Lesquelles un test sur des données « normales » ne voit-il pas ?

In [ ]:
rng_17 = np.random.default_rng(1717)
X_17 = rng_17.normal(size=(8, 3))
PARAMS_17 = [(rng_17.normal(size=(3, 4)), rng_17.normal(size=4)), (rng_17.normal(size=(4, 3)), rng_17.normal(size=3))]
BIG_17 = [PARAMS_17[0], (400.0 * PARAMS_17[1][0], 400.0 * PARAMS_17[1][1])]      # logits of order 1000
Y_17 = np.array([0, 1, 2, 0, 1, 2, 0, 1])


def reference_17(X, y, params):
    """What forward_17 must return, computed with SciPy (the oracle of the diagnostics)."""
    (W1, b1), (W2, b2) = params
    z1 = X @ W1 + b1
    logits = np.tanh(z1) @ W2 + b2
    return special.softmax(logits, axis=1), float(-np.mean(special.log_softmax(logits, axis=1)[np.arange(len(y)), y])), 1 - np.tanh(z1) ** 2

In [ ]:
def forward_17(X, y, params):
    """Probabilities, mean cross-entropy and local slopes of a 2-layer tanh network (one row per sample).

    params = [(W1, b1), (W2, b2)], W of shape (n_in, n_out); y holds the class of each row (0 to K - 1).
    Returns (proba of shape (n, K), loss as a float, slope = tanh'(z1) of shape (n, n_hidden)).
    """
    (W1, b1), (W2, b2) = params
    z1 = X @ W1 + b1
    h = np.tanh(z1)
    slope = 1.0 - np.tanh(h) ** 2                     # tanh'(z1), for the backward pass of chapter 18
    logits = h @ W2 + b2
    e = np.exp(logits)
    proba = e / e.sum(axis=0, keepdims=True)
    loss = -np.mean(np.log(proba[np.arange(len(y)), y]))
    return proba, loss, slope

In [ ]:
def diagnose_17(forward):
    """The four points of 17.17, in the order in which the bugs hide each other; returns the first failing point (or 0)."""
    points = [
        (1, "aucun débordement (« overflow ») de l'exponentielle pour des logits de l'ordre de 1 000"),
        (2, "chaque LIGNE de probabilités somme à 1 et vaut la softmax de SciPy"),
        (3, "une loss finie, égale à la cross-entropy de SciPy, quand la bonne classe a une probabilité qui s'arrondit à 0"),
        (4, "slope vaut tanh'(z1), la dérivée prise en la pré-activation"),
    ]
    for k, what in points:
        try:
            if k == 1:
                (proba, _, _), alarms = quiet(forward, X_17, Y_17, BIG_17)
                ok = not [a for a in alarms if "overflow" in a]      # nan from the axis is point 2, inf from the log point 3
            elif k == 2:
                (proba, _, _), _ = quiet(forward, X_17, Y_17, BIG_17)      # its warnings belong to the other points
                want, _, _ = reference_17(X_17, Y_17, BIG_17)
                ok = np.shape(proba) == want.shape and bool(np.allclose(proba, want, atol=1e-12))
            elif k == 3:
                logits = np.tanh(X_17 @ BIG_17[0][0] + BIG_17[0][1]) @ BIG_17[1][0] + BIG_17[1][1]
                y_worst = np.argmin(logits, axis=1)               # the least likely class of each row
                (_, loss, _), alarms = quiet(forward, X_17, y_worst, BIG_17)
                _, want, _ = reference_17(X_17, y_worst, BIG_17)
                ok = not alarms and np.isfinite(float(loss)) and bool(np.isclose(float(loss), want, rtol=1e-9))
            else:
                (_, _, slope), _ = quiet(forward, X_17, Y_17, PARAMS_17)
                _, _, want = reference_17(X_17, Y_17, PARAMS_17)
                ok = np.shape(slope) == want.shape and bool(np.allclose(slope, want, atol=1e-12))
        except (NotImplementedError, MylearnMissingError):
            raise
        except Exception as exc:                                # a crash counts as a failed point
            ok, what = False, f"{what} (ton code lève {type(exc).__name__} : {str(exc)[:120]})"
        verdict("17.17", ok, f"point {k} : {what}.", f"point {k} : attendu : {what}.")
        if not ok:
            return k
    return 0


with wb.attempt("17.17"):
    first_17 = diagnose_17(forward_17)
    if first_17:
        print(f"⏳ Ex 17.17 : corrige d'abord le point {first_17} ; les points suivants seront vérifiés ensuite.")
    else:
        print("✅ Ex 17.17 : les quatre points sont corrigés.")

## Partie C · Comparer les activations

Refaire la galerie avec ton code et y mesurer quatre valeurs (17.18), prédire l'effet d'une température (17.19), comparer les quatre activations de scikit-learn sur deux problèmes (17.20), puis lire des histogrammes de pré-activations (17.21).

### Ex 17.18 — Reproduire la galerie de la figure 17.22, dérivées comprises 🎨 ★★ ⏱️ 25 min
**Objectif :** tracer les dix activations de ta librairie avec leurs dérivées, et lire sur des grilles fines quatre points remarquables.  
**Prérequis :** Ex 17.14 · fiche §17.7 · **Parcours :** C

Refais la galerie de la fiche (§17.7) avec **ta** librairie : une grille de 2 × 5 graphiques, un par nom de `NAMES_18`, avec `f` en trait plein et `f′` en tirets (obtenus par `get_activation`), pour $z$ de $-4$ à $4$ et une ordonnée de $-1{,}5$ à $2{,}5$. Puis, avec tes fonctions, calcule sur des grilles **fines** (un pas de $10^{-4}$ ou moins) :
a) `zmin_silu_18` : la valeur de $z$ où SiLU est la plus petite ;
b) `min_gelu_18` : la plus petite valeur de GELU ;
c) `max_dgelu_18` : la plus grande valeur de la dérivée de GELU ;
d) `zmax_dsilu_18` : la valeur de $z > 0$ où la dérivée de SiLU est la plus grande.

Garde les valeurs telles quelles : la vérification arrondit elle-même (2 décimales pour a) et d), 3 pour b) et c)).

Dans tes notes : quelles fonctions de la galerie ont une dérivée qui dépasse 1 ? Qu'est-ce que cela change pour le produit des dérivées de la rétropropagation, par rapport à la sigmoïde ?

In [ ]:
# 1) The gallery: a 2 x 5 grid, one panel per activation of get_activation, f in a solid line and f' dashed,
#    on z from -4 to 4, with the same axis limits as the figure of the sheet.
NAMES_18 = ["identity", "step", "relu", "leaky_relu", "elu", "softplus", "sigmoid", "tanh", "silu", "gelu"]
# your plotting code here, then plt.show()

# 2) Four values read on fine grids with your functions (do not round them)
zmin_silu_18 = ...      # a) the z where SiLU is smallest
min_gelu_18 = ...       # b) the smallest value of GELU
max_dgelu_18 = ...      # c) the largest value of the derivative of GELU
zmax_dsilu_18 = ...     # d) the z > 0 where the derivative of SiLU is largest

In [ ]:
wb.check("17.18a", zmin_silu_18)
wb.check("17.18b", min_gelu_18)
wb.check("17.18c", max_dgelu_18)
wb.check("17.18d", zmax_dsilu_18)

### Ex 17.19 — Softmax et température : prédire la distribution 🔮 ★★ ⏱️ 15 min
**Objectif :** prédire l'effet d'une température sur une distribution de probabilités, chiffres à l'appui.  
**Prérequis :** Ex 17.15 · ch. 6 (entropie) · fiche §17.8 (la température) · **Parcours :** M, C

Un classifieur à cinq classes, numérotées de 0 à 4, sort les scores `SCORES_19` $= (3 ;\ 1{,}5 ;\ 1 ;\ 0 ;\ -1)$. On calcule $\mathrm{softmax}(\mathbf{z}/T)$ avec ta `softmax`, pour plusieurs températures $T$, et l'entropie de chaque distribution, en bits (ch. 6 ; son maximum, pour cinq classes, vaut $\log_2 5 \approx 2{,}32$).

Prédis, **avant** d'exécuter quoi que ce soit :
a) `sharp_19` : à $T = 0{,}5$, la classe en tête a-t-elle une probabilité supérieure à 0,9 ? (`True` ou `False`) ;
b) `flat_19` : à $T = 5$, la classe la moins probable a-t-elle une probabilité supérieure à 0,1 ? (`True` ou `False`) ;
c) `half_19` : parmi les températures (A) 0,25 ; (B) 0,5 ; (C) 1 ; (D) 2 ; (E) 4 ; (F) 8, la première dont l'entropie dépasse la moitié du maximum (la lettre) ;
d) `negative_19` : avec une température **négative**, $T = -1$, le numéro de la classe en tête (un entier de 0 à 4).

Puis exécute l'expérience : elle affiche les distributions et leurs entropies, trace l'entropie en fonction de $T$, et vérifie tes prédictions.

In [ ]:
SCORES_19 = np.array([3.0, 1.5, 1.0, 0.0, -1.0])     # five classes, numbered 0 to 4
TEMPERATURES_19 = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
sharp_19 = ...      # a) True or False
flat_19 = ...       # b) True or False
half_19 = ...       # c) "A" to "F"
negative_19 = ...   # d) an integer from 0 to 4

In [ ]:
if any(answer is ... or answer is None for answer in [sharp_19, flat_19, half_19, negative_19]):
    print("⏳ Ex 17.19 : écris d'abord tes quatre prédictions, puis relance cette cellule.")
else:
    with wb.attempt("17.19"):
        A = mylearn.nn.activations
        table_19 = {T: np.asarray(A.softmax(SCORES_19 / T), dtype=float) for T in TEMPERATURES_19 + [5.0, -1.0]}
        entropy_19 = {T: float(-(p * np.log2(p)).sum()) for T, p in table_19.items()}
        for T in TEMPERATURES_19 + [5.0, -1.0]:
            print(f"T = {T:5.2f} : p = {np.round(table_19[T], 3).tolist()} · entropy {entropy_19[T]:.3f} bits "
                  f"(maximum log2(5) = {np.log2(5):.3f})")
        fig_19, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(11, 3.6))
        for T in [0.5, 1.0, 5.0]:
            ax_a.plot(range(5), table_19[T], "o-", label=f"T = {T}")
        ax_a.set_xticks(range(5))
        ax_a.set_xlabel("classe")
        ax_a.set_ylabel("probabilité")
        ax_a.legend()
        ax_b.plot(TEMPERATURES_19, [entropy_19[T] for T in TEMPERATURES_19], "o-")
        ax_b.axhline(np.log2(5) / 2, color="grey", ls="--", label="la moitié du maximum")
        ax_b.set_xscale("log")
        ax_b.set_xlabel("température T (échelle log)")
        ax_b.set_ylabel("entropie (bits)")
        ax_b.legend()
        plt.tight_layout()
        plt.show()
    wb.check("17.19a", sharp_19)
    wb.check("17.19b", flat_19)
    wb.check("17.19c", half_19)
    wb.check("17.19d", negative_19)

Dans tes notes : comment l'entropie évolue-t-elle quand $T$ grandit, et vers quelle valeur tend-elle ? Que fait une température négative à l'ordre des classes ? Un modèle de langage « créatif » utilise-t-il une petite ou une grande température ?

### Ex 17.20 — Quatre activations de scikit-learn sur les lunes et les spirales 📦 ★★ ⏱️ 25 min
**Objectif :** comparer, avec un protocole reproductible, les quatre activations de `MLPClassifier` sur deux problèmes.  
**Prérequis :** Ex 17.11 · ch. 15 (`MLPClassifier`) · fiche §17.6, section sur le choix d'une activation · **Fil rouge :** synthétique · **Parcours :** R, C

`MLPClassifier` propose quatre activations cachées : `"identity"`, `"logistic"` (la sigmoïde), `"tanh"` et `"relu"`. Compare-les sur deux problèmes : les deux lunes (600 points, bruit 0,25) et deux spirales (600 points, bruit 0,2, 1,5 tour), avec 30 % pour le test (`DATA_20`).

Écris `mean_accuracy_20(dataset, activation)` : pour `random_state` valant 0, 1, 2, 3 et 4, entraîne `MLPClassifier((16,), activation=activation, solver="lbfgs", max_iter=3000, random_state=random_state)` sur la partie d'entraînement de `DATA_20[dataset]` (avec `fit_quietly`), mesure son accuracy de test, et rends la **moyenne** des cinq accuracies (un `float`). La vérification calcule les huit moyennes avec ta fonction, affiche le tableau, et vérifie :
a) la moyenne de l'identité sur les spirales ;
b) `best_moons_20` : l'activation de meilleure moyenne sur les lunes (son nom, entre guillemets) ;
c) `best_spirals_20` : l'activation de meilleure moyenne sur les spirales ;
d) l'écart entre la moyenne de ReLU et celle de tanh sur les spirales (ReLU moins tanh).

Remplis b) et c) **après** avoir lu le tableau.

Dans tes notes : le classement est-il le même sur les deux problèmes ? Explique l'écart entre les activations sur les spirales (pense à la forme de la frontière que 16 neurones de chaque sorte peuvent tracer). Que dit l'écart entre les cinq graines (relance une ligne avec `np.std`) de la fiabilité d'une comparaison sur une seule graine ?

In [ ]:
def split_20(X, y):
    return train_test_split(X, y, test_size=0.3, random_state=1720, stratify=y)


DATA_20 = {"lunes": split_20(*wb.synth.make_moons(n=600, noise=0.25, seed=1720)),
           "spirales": split_20(*wb.synth.make_spirals(n=600, noise=0.2, turns=1.5, seed=1720))}
ACTIVATIONS_20 = ["identity", "logistic", "tanh", "relu"]
for name, (Xtr, Xte, ytr, yte) in DATA_20.items():
    print(name, "train", Xtr.shape, "test", Xte.shape)

In [ ]:
def mean_accuracy_20(dataset, activation):
    """Mean test accuracy over random_state = 0, 1, 2, 3, 4 of the protocol of the statement."""
    raise NotImplementedError("mean_accuracy_20")


best_moons_20 = ...      # b) the best activation on the moons, e.g. "identity"
best_spirals_20 = ...    # c) the best activation on the spirals

In [ ]:
results_20 = {}
with checked("17.20"):
    for dataset_20 in DATA_20:
        results_20[dataset_20] = {a: mean_accuracy_20(dataset_20, a) for a in ACTIVATIONS_20}
        print(f"{dataset_20:9s}", " · ".join(f"{a} {v:.3f}" for a, v in results_20[dataset_20].items()))
if len(results_20) == 2:
    wb.check("17.20a", results_20["spirales"]["identity"], computed=True)
    wb.check("17.20d", results_20["spirales"]["relu"] - results_20["spirales"]["tanh"], computed=True)
with wb.attempt("17.20"):
    wb.check("17.20b", best_moons_20)
    wb.check("17.20c", best_spirals_20)

### Ex 17.21 — Diagnostiquer la saturation sur des histogrammes de pré-activations 📈 ★★ ⏱️ 20 min
**Objectif :** repérer sur des histogrammes les couches sigmoïdes saturées, puis le chiffrer avec ta dérivée.  
**Prérequis :** Ex 17.13 · ch. 16 (histogrammes d'activations) · fiche §17.6 (la saturation) · **Fil rouge :** synthétique · **Parcours :** M

La cellule ci-dessous trace les pré-activations $z$ de quatre couches sigmoïdes P, Q, R et S (1 000 exemples, 5 neurones chacune), comme le ferait un diagnostic avant l'entraînement (ch. 16). En rose : la zone $|z| > 4{,}6$, où $\sigma'(z) < 0{,}01$.

En lisant les histogrammes :
a) `most_21` : la lettre de la couche la plus saturée (celle dont les neurones ont, en moyenne, la plus petite pente) ;
b) `linear_21` : la lettre de la couche qui travaille presque dans la partie linéaire de la sigmoïde (une pente presque constante).

Puis écris `saturation_21(z)`, qui rend, avec **ta** `sigmoid_derivative`, la pente moyenne $\overline{\sigma'(z)}$ et la part des valeurs où $\sigma'(z) < 0{,}01$. La vérification calcule :
c) les pentes moyennes des quatre couches, dans l'ordre P, Q, R, S (une liste) ;
d) la part des pentes inférieures à 0,01 dans la couche R.

Dans tes notes : la couche R a-t-elle plus de valeurs dans la zone rose que Q ? Pourquoi sa pente moyenne est-elle pourtant plus grande ? Qu'est-ce qui, dans une initialisation ou dans les données, peut produire chacun des quatre histogrammes ?

In [ ]:
rng_21 = np.random.default_rng(1721)
SPECS_21 = {"P": (0.0, 1.0), "Q": (4.0, 1.0), "R": (0.0, 4.0), "S": (-1.0, 0.3)}       # mean, standard deviation
Z_21 = {name: rng_21.normal(m, s, size=(1000, 5)) for name, (m, s) in SPECS_21.items()}  # 1000 samples, 5 neurons
fig_21, axes_21 = plt.subplots(1, 4, figsize=(14, 3.0), sharex=True)
for ax, (name, z) in zip(axes_21, Z_21.items()):
    ax.hist(z.ravel(), bins=60, range=(-14, 14), color="#2a78d6")
    ax.axvspan(-14, -4.6, color="#f4a3a2", alpha=0.3)
    ax.axvspan(4.6, 14, color="#f4a3a2", alpha=0.3)
    ax.set_title(f"couche {name}")
    ax.set_xlabel("pré-activation z")
fig_21.suptitle("Pré-activations de quatre couches sigmoïdes (en rose : |z| > 4,6, où σ′(z) < 0,01)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
most_21 = ...       # a) the letter of the most saturated layer, read on the histograms
linear_21 = ...     # b) the letter of the layer that works in the almost linear part of the sigmoid


def saturation_21(z):
    """(mean of sigmoid'(z), fraction of the values where sigmoid'(z) < 0.01), with YOUR sigmoid_derivative."""
    raise NotImplementedError("saturation_21")

In [ ]:
with wb.attempt("17.21"):
    wb.check("17.21a", most_21)
    wb.check("17.21b", linear_21)
with checked("17.21"):
    stats_21 = {name: saturation_21(z) for name, z in Z_21.items()}
    for name, (mean_slope, frac) in stats_21.items():
        print(f"couche {name} : pente moyenne {mean_slope:.4f} · part des pentes < 0,01 : {frac:.3f}")
    wb.check("17.21c", [stats_21[name][0] for name in "PQRS"], computed=True)
    wb.check("17.21d", stats_21["R"][1], computed=True)

## Partie D · Tester et défier

Écrire une batterie de tests paramétrés pour toute une famille de fonctions (17.22), mesurer ce que chaque activation fait à un gradient qui traverse dix couches (17.23), puis relever le défi des spirales sous contrainte de paramètres (17.24).

### Ex 17.22 — Une batterie de tests paramétrés pour toutes les activations 🛠️ ★★ ⏱️ 25 min
**Objectif :** écrire des tests pytest paramétrés et des tests de propriétés qui attrapent cinq bugs différents dans une famille de fonctions.  
**Prérequis :** Ex 17.14 · 0A (`pytest.mark.parametrize`, `wb.run_pytest`) · fiche, section sur le choix et les tests · **Parcours :** C

La fonction `act_ok(name)` (fournie) rend le couple `(f, f′)` de quatre activations, `"relu"`, `"sigmoid"`, `"tanh"` et `"softplus"`, avec le contrat de ta librairie : chaque fonction rend un nouveau tableau `float64`, laisse son entrée intacte, reste finie sans avertissement pour $|z| \le 1\,000$, chaque dérivée vaut la pente de sa fonction, et $\mathrm{ReLU}'(0) = 0$ comme dans PyTorch. Écris une **batterie de tests** pytest pour une fonction `act(name)` qui a ce contrat : au moins un test doit être **paramétré** (`@pytest.mark.parametrize`, par exemple sur les quatre noms), et tu peux tester des **propriétés** (une fonction croissante, une dérivée égale à la pente numérique) plutôt que des valeurs.

Range tes tests dans la liste `my_tests_22`. Comme en 16.21, la vérification lance vraiment pytest : tes tests doivent **passer** sur `act_ok` et **échouer** sur chacune des cinq versions buggées fournies, `act_bug_1` à `act_bug_5` (lis-les : que fait chacune de travers ?). Tes tests ne peuvent utiliser que `act`, `np`, `math` et `pytest`. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque version buggée, le test qui l'attrape. Lesquelles passeraient un test qui compare les valeurs en trois points « ordinaires » comme $-1$, $0$ et $2$ ?

In [ ]:
import math

import pytest                                     # for the @pytest.mark.parametrize decorators of your tests


def act_ok(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": four activations and their derivatives (correct)."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z > 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

    def sigmoid_derivative(z):
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.array(z, dtype=float)) ** 2

    def softplus(z):
        z = np.array(z, dtype=float)
        return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]


def act_bug_1(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": the same, with one bug."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z >= 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

    def sigmoid_derivative(z):
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.array(z, dtype=float)) ** 2

    def softplus(z):
        z = np.array(z, dtype=float)
        return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]


def act_bug_2(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": the same, with one bug."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z > 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, e / (1 + e), 1 / (1 + e))

    def sigmoid_derivative(z):
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.array(z, dtype=float)) ** 2

    def softplus(z):
        z = np.array(z, dtype=float)
        return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]


def act_bug_3(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": the same, with one bug."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z > 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

    def sigmoid_derivative(z):
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.tanh(np.array(z, dtype=float))) ** 2

    def softplus(z):
        z = np.array(z, dtype=float)
        return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]


def act_bug_4(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": the same, with one bug."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z > 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

    def sigmoid_derivative(z):
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.array(z, dtype=float)) ** 2

    def softplus(z):
        return np.log1p(np.exp(np.array(z, dtype=float)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]


def act_bug_5(name):
    """(f, f') for "relu", "sigmoid", "tanh" or "softplus": the same, with one bug."""
    def relu(z):
        z = np.array(z, dtype=float)
        return np.where(z > 0, z, 0.0)

    def relu_derivative(z):
        z = np.array(z, dtype=float)
        return (z > 0).astype(float)

    def sigmoid(z):
        z = np.array(z, dtype=float)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

    def sigmoid_derivative(z):
        z = np.asarray(z, dtype=float)
        z *= -1.0                       # sigma'(-z) = sigma'(z): "saves" a copy
        s = sigmoid(z)
        return s * (1 - s)

    def tanh(z):
        return np.tanh(np.array(z, dtype=float))

    def tanh_derivative(z):
        return 1 - np.tanh(np.array(z, dtype=float)) ** 2

    def softplus(z):
        z = np.array(z, dtype=float)
        return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))

    def softplus_derivative(z):
        return sigmoid(z)

    return {"relu": (relu, relu_derivative), "sigmoid": (sigmoid, sigmoid_derivative),
            "tanh": (tanh, tanh_derivative), "softplus": (softplus, softplus_derivative)}[name]

In [ ]:
# Write your test functions here (their names start with test_). They test a function act(name) whose contract
# is the one of act_ok (read it): act(name) returns the pair (f, f'). At least ONE of your tests must be
# parametrized with @pytest.mark.parametrize, for example over the four names:
#
# @pytest.mark.parametrize("name", ["relu", "sigmoid", "tanh", "softplus"])
# def test_something(name):
#     f, df = act(name)
#     assert ...

my_tests_22 = ...   # the list of your test functions

In [ ]:
def parametrized_22(tests):
    """True when at least one test function carries a pytest.mark.parametrize mark."""
    return any(mark.name == "parametrize" for t in tests for mark in getattr(t, "pytestmark", []))


if my_tests_22 is ...:
    print("⏳ Ex 17.22 : tests pas encore écrits.")
else:
    good_22 = wb.run_pytest(my_tests_22, subject=act_ok, name="act")
    verdict("17.22", parametrized_22(my_tests_22), "au moins un de tes tests est paramétré.",
            "aucun de tes tests n'est paramétré : utilise @pytest.mark.parametrize pour en exécuter un sur toute la famille.")
    verdict("17.22", good_22.ok, f"tes tests passent sur la version juste ({good_22.passed} cas).",
            "tes tests doivent tous passer sur act_ok, la version juste (voir le détail au-dessus).")
    if good_22.ok:
        for bug_22 in [act_bug_1, act_bug_2, act_bug_3, act_bug_4, act_bug_5]:
            result_22 = wb.run_pytest(my_tests_22, subject=bug_22, name="act", quiet=True)
            verdict("17.22", result_22.failed + result_22.errors > 0, f"{bug_22.__name__} est attrapé ({result_22.failed} cas en échec).",
                    f"tes tests passent sur {bug_22.__name__}, qui est faux : lis-le, puis ajoute un test qui le fait échouer.")

### Ex 17.23 — Produit des dérivées sur 10 couches : sigmoïde, tanh, ReLU, GELU 🔬 ★★★ ⏱️ 45 min
**Objectif :** mesurer, couche par couche, les pentes des activations d'un réseau profond et ce qu'il reste d'un gradient après dix couches.  
**Prérequis :** Ex 17.14 · ch. 16 (initialisations de Glorot et He) · fiche §17.6 (la saturation) · **Fil rouge :** synthétique · **Parcours :** R, M, C

Un réseau de dix couches carrées de 64 neurones, sans biais, reçoit 500 exemples standardisés (`X_23`). Ses poids viennent de `weights_23(name)` : Glorot pour la sigmoïde et tanh, He pour ReLU et GELU, comme le recommande le ch. 16. Écris `slopes_23(name)` : la passe avant de `X_23` à travers ces dix couches ($\mathbf{z} = \mathbf{a}\mathbf{W}$, puis $\mathbf{a} = f(\mathbf{z})$), qui rend la liste des dix tableaux $f'(\mathbf{z})$, avec $f$ et $f'$ tirés de **ton** `get_activation`.

La vérification calcule, pour chaque activation, la pente moyenne de chaque couche, le **produit** de ces dix pentes moyennes, et le rapport entre la norme du gradient qui arrive sur l'entrée et celle du gradient parti de la dernière couche, calculé par `backward_ratio_23` avec la règle de la chaîne (le ch. 18 la détaillera : à chaque couche, le gradient est multiplié case par case par $f'(\mathbf{z})$, puis par $\mathbf{W}^\top$). Elle trace le produit cumulé des pentes moyennes, et contrôle trois ordres de grandeur.

Dans tes notes : pour quelle activation le gradient s'évanouit-il ? Pourquoi le produit des pentes moyennes de ReLU est-il si petit, alors que son gradient arrive presque intact ? Que se passerait-il avec ReLU et l'initialisation de Glorot (copie `weights_23` avec un paramètre `gain`, et refais l'expérience) ?

In [ ]:
X_23 = np.random.default_rng(1723).normal(size=(500, 64))      # 500 standardized samples, 64 features
G_23 = np.random.default_rng(17231).normal(size=(500, 64))      # a gradient arriving on the last layer


def weights_23(name):
    """Ten square 64 x 64 layers, without bias: Glorot for sigmoid and tanh, He for relu and gelu (chapter 16)."""
    gain = 2.0 if name in ("relu", "gelu") else 1.0
    rng = np.random.default_rng(17230)
    return [rng.normal(0.0, np.sqrt(gain / 64), size=(64, 64)) for _ in range(10)]


def backward_ratio_23(Ws, slopes):
    """||gradient on the input|| / ||gradient on the last layer||, by the chain rule through the ten layers (ch. 18)."""
    g = G_23
    for W, s in zip(reversed(Ws), reversed(slopes)):
        g = (g * s) @ W.T
    return float(np.linalg.norm(g) / np.linalg.norm(G_23))

In [ ]:
def slopes_23(name):
    """The ten arrays f'(z) (each of shape (500, 64)) of a forward pass of X_23 through weights_23(name),
    a = f(z) with z = a @ W at each layer, f and f' from YOUR get_activation(name)."""
    raise NotImplementedError("slopes_23")

In [ ]:
summary_23 = {}
with checked("17.23"):
    for name_23 in ["sigmoid", "tanh", "relu", "gelu"]:
        s_23 = slopes_23(name_23)
        if len(s_23) != 10 or any(np.shape(s) != (500, 64) for s in s_23):
            print(f"❌ Ex 17.23 : slopes_23({name_23!r}) doit rendre 10 tableaux de forme (500, 64).")
            continue
        means_23 = [float(np.mean(s)) for s in s_23]
        summary_23[name_23] = (means_23, float(np.prod(means_23)), backward_ratio_23(weights_23(name_23), s_23))
        print(f"{name_23:8s} mean f' per layer {np.round(means_23, 3).tolist()}")
        print(f"{'':8s} product of the means {summary_23[name_23][1]:.2e} · gradient ratio input/output {summary_23[name_23][2]:.2e}")
if len(summary_23) == 4:
    fig_23, ax_23 = plt.subplots(figsize=(7, 3.6))
    for name_23, (means_23, _, _) in summary_23.items():
        ax_23.plot(range(1, 11), np.cumprod(means_23), "o-", label=name_23)
    ax_23.set_yscale("log")
    ax_23.set_xlabel("couche")
    ax_23.set_ylabel("produit des pentes moyennes")
    ax_23.legend()
    plt.tight_layout()
    plt.show()
    verdict("17.23", summary_23["sigmoid"][2] < 1e-4, f"sigmoïde : le gradient est divisé par plus de 10⁴ ({summary_23['sigmoid'][2]:.1e}).",
            f"sigmoïde : un rapport de {summary_23['sigmoid'][2]:.1e} est inattendu ; vérifie tes dérivées (σ′ ≤ 1/4).")
    verdict("17.23", 0.1 < summary_23["relu"][2] < 10, f"ReLU avec He : le gradient garde son ordre de grandeur ({summary_23['relu'][2]:.2f}).",
            f"ReLU avec He : un rapport de {summary_23['relu'][2]:.2e} est inattendu ; vérifie relu_derivative.")
    verdict("17.23", summary_23["relu"][1] < 1e-2 < summary_23["relu"][2],
            "ReLU : le produit des pentes moyennes s'effondre, mais pas le gradient : les poids de He compensent.",
            "le produit des pentes de ReLU devrait être petit (≈ 0,5 par couche) et le rapport de gradients proche de 1.")

### Ex 17.24 — Défi spirales : au moins 97 % avec au plus 2 000 paramètres 🏆 ★★★ ⏱️ 60 min
**Objectif :** choisir activation, architecture, solveur et régularisation d'un MLP sous une contrainte de paramètres, et le vérifier sur des données jamais vues.  
**Prérequis :** Ex 17.20 · ch. 16 (compter les paramètres) · ch. 8 (validation croisée) · ch. 15 (`Pipeline`) · **Fil rouge :** synthétique · **Parcours :** C

Deux spirales entrelacées de 1 600 points (bruit 0,3, 2,5 tours) : la moitié pour l'entraînement (`X_24`, `y_24`), l'autre moitié forme un **test caché**. Écris `build_24()`, qui rend un modèle scikit-learn **neuf** (pas encore entraîné) : un `MLPClassifier`, éventuellement précédé d'un prétraitement dans un `Pipeline` (ch. 15). Tu choisis l'activation (parmi les quatre de scikit-learn), l'architecture, le solveur, la régularisation (`alpha`) et le nombre d'itérations.

Règles : au plus **2 000 paramètres** entraînés (poids et biais du `MLPClassifier`, comptés par `n_params_24`) ; le défi est réussi avec une accuracy d'au moins **0,97** sur le test caché **et** en moyenne sur quatre autres tirages des spirales (ta méthode est réentraînée sur chaque tirage) ; palier 🌟 : au moins 0,99 sur le test caché. La vérification entraîne ton modèle une dizaine de fois (validation croisée, test caché, quatre tirages) : garde un entraînement sous 5 secondes environ (`max_iter` de 1 000 à 3 000). Règle d'honneur : choisis tes réglages par validation croisée sur `X_24` seulement (la vérification l'affiche), et ne regarde le test caché qu'une fois : la vérification ne le révèle qu'une fois par session, et fige le résultat.

Dans tes notes : ta démarche (ce que tu as essayé, dans quel ordre, et les scores de validation croisée), puis ce qui a le plus compté : l'activation, la taille, `alpha` ou le solveur ? Le modèle par défaut, `MLPClassifier()`, qu'obtient-il en validation croisée ?

In [ ]:
def spirals_24(seed):
    """Two spirals of 1600 points (noise 0.3, 2.5 turns), half for training and half for testing."""
    X, y = wb.synth.make_spirals(n=1600, n_classes=2, noise=0.3, turns=2.5, seed=seed)
    return train_test_split(X, y, test_size=0.5, random_state=seed, stratify=y)


X_24, X_HIDDEN_24, y_24, Y_HIDDEN_24 = spirals_24(1724)          # the test half is the hidden test: do not look at it
OTHER_SEEDS_24 = [2401, 2402, 2403, 2404]
print("training points:", X_24.shape, "· classes:", np.bincount(y_24).tolist())


def n_params_24(model):
    """The number of trained parameters of the MLPClassifier of a fitted model (or of the last step of a Pipeline)."""
    mlp = model[-1] if hasattr(model, "steps") else model
    return sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)

In [ ]:
def build_24():
    """A NEW, unfitted scikit-learn model: an MLPClassifier, or a Pipeline whose last step is an MLPClassifier."""
    raise NotImplementedError("build_24")

In [ ]:
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import cross_val_score
from sklearn.utils.validation import check_is_fitted

with checked("17.24"):
    model_24 = build_24()
    last_24 = model_24[-1] if hasattr(model_24, "steps") else model_24
    rules_24 = isinstance(last_24, MLPClassifier)
    verdict("17.24", rules_24, "le modèle se termine par un MLPClassifier.",
            f"le dernier élément du modèle est un {type(last_24).__name__} : le défi demande un MLPClassifier.")
    try:
        check_is_fitted(model_24)
        rules_24 = False
        print("❌ Ex 17.24 : build_24 rend un modèle déjà entraîné : rends un modèle neuf.")
    except NotFittedError:
        pass
    if build_24() is model_24:
        rules_24 = False
        print("❌ Ex 17.24 : build_24 rend deux fois le même objet : crée un nouveau modèle à chaque appel.")
    if rules_24:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            cv_24 = cross_val_score(clone(model_24), X_24, y_24, cv=5)
        fitted_24 = fit_quietly(build_24(), X_24, y_24)
        params_24 = n_params_24(fitted_24)
        print(f"validation croisée sur l'entraînement : {cv_24.mean():.3f} ± {cv_24.std():.3f} · {params_24} paramètres")
        verdict("17.24", params_24 <= 2000, f"{params_24} paramètres, sous la limite de 2 000.",
                f"{params_24} paramètres : la limite est de 2 000 (poids et biais du MLPClassifier).")
        if params_24 <= 2000:
            if globals().get("REVEALED_24") is None:              # revealed once per session (an honour rule)
                others_24 = []
                for seed_24 in OTHER_SEEDS_24:
                    Xa, Xb, ya, yb = spirals_24(seed_24)
                    others_24.append(fit_quietly(build_24(), Xa, ya).score(Xb, yb))
                REVEALED_24 = (fitted_24.score(X_HIDDEN_24, Y_HIDDEN_24), float(np.mean(others_24)), params_24)
            test_24, mean_24, frozen_params_24 = REVEALED_24
            print(f"test caché : {test_24:.3f} · moyenne sur quatre autres tirages : {mean_24:.3f} "
                  f"(révélés une seule fois par session, pour le modèle à {frozen_params_24} paramètres)")
            verdict("17.24", test_24 >= 0.97 and mean_24 >= 0.97,
                    "défi réussi : au moins 0,97 sur le test caché et en moyenne sur quatre autres tirages."
                    + (" 🌟 Palier atteint : au moins 0,99 sur le test caché." if test_24 >= 0.99 else ""),
                    "pas encore : il faut au moins 0,97 sur le test caché ET en moyenne sur les quatre autres tirages.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu montrer qu'un réseau sans activation s'effondre, et calculer la couche qui le remplace ?
2. Sais-tu écrire chaque activation et sa dérivée sans débordement, et la softmax, la log-softmax et sa Jacobienne ?
3. Sais-tu choisir l'activation des couches cachées et celle de la sortie, et justifier ce choix par la saturation, les neurones morts et les gradients ?

**Pour aller plus loin** : les articles de Hendrycks et Gimpel (2016), de Ramachandran, Zoph et Le (2017) et de Shazeer (2020), cités dans la fiche. La suite : le ch. 18, où les couples `(f, f′)` de ton `get_activation` et le cache de ton `mlp_forward` serviront à la rétropropagation.